# 1391. Check if There is a Valid Path in a Grid

## Topic Alignment
- **Role Relevance**: Model directional connectivity in transportation networks where path validity depends on directional constraints.
- **Scenario**: Analyze street networks with one-way roads, pipeline systems with flow direction, or circuit design with directional components.

## Metadata Summary
- Source: [Check if There is a Valid Path in a Grid](https://leetcode.com/problems/check-if-there-is-a-valid-path-in-a-grid/)
- Tags: `Union-Find`, `BFS`, `DFS`, `Array`, `Matrix`
- Difficulty: Medium
- Recommended Priority: High

## Problem Statement
You are given an `m x n` grid. Each cell of the grid represents a street. The street of `grid[i][j]` can be:
- `1` which means a street connecting the left cell and the right cell.
- `2` which means a street connecting the upper cell and the lower cell.
- `3` which means a street connecting the left cell and the lower cell.
- `4` which means a street connecting the right cell and the lower cell.
- `5` which means a street connecting the left cell and the upper cell.
- `6` which means a street connecting the right cell and the upper cell.

You will initially start at the street of the upper-left cell `(0, 0)`. A valid path in the grid is a path that starts from the upper left cell `(0, 0)` and ends at the bottom-right cell `(m - 1, n - 1)`. The path should only follow the streets.

Return `true` if there is a valid path in the grid or `false` otherwise.

**Example**:
```
Input: grid = [[2,4,3],[6,5,2]]
Output: true
```

## Progressive Hints
- Hint 1: Each street type has specific entry and exit points - model these as directional connections.
- Hint 2: Two cells can be connected only if one cell's exit aligns with the adjacent cell's entry.
- Hint 3: Use Union-Find to group all validly connected cells, then check if start and end are in the same component.
- Hint 4: Define direction mappings: LEFT, RIGHT, UP, DOWN, and check compatibility between adjacent cells.

## Solution Overview
Use Union-Find to connect cells that have compatible directional streets. Check if the top-left cell `(0, 0)` and bottom-right cell `(m-1, n-1)` are in the same connected component.

## Detailed Explanation
1. **Street Type Mapping**: Each street type connects specific directions:
   - Type 1: LEFT ↔ RIGHT (horizontal)
   - Type 2: UP ↔ DOWN (vertical)
   - Type 3: LEFT ↔ DOWN (L-shape)
   - Type 4: RIGHT ↔ DOWN (inverted L)
   - Type 5: LEFT ↔ UP (reverse L)
   - Type 6: RIGHT ↔ UP (reverse inverted L)
2. **Direction Compatibility**: Define which directions can connect:
   - If current cell can go RIGHT, next cell must accept LEFT
   - If current cell can go DOWN, next cell must accept UP
   - Similar logic for LEFT and UP directions
3. **Union-Find Construction**: For each cell, check all 4 adjacent cells:
   - If current cell's exit direction matches adjacent cell's entry direction
   - Union these two cells in the same component
4. **Path Validation**: After building all connections, check if `find(0, 0) == find(m-1, n-1)`
5. **Edge Cases**: Single cell grid (already connected), disconnected streets, incompatible street types.

## Complexity Trade-off Table
| Approach | Time Complexity | Space Complexity | Notes |
| --- | --- | --- | --- |
| Union-Find | O(m*n α(m*n)) | O(m*n) | Optimal for connectivity queries |
| BFS | O(m*n) | O(m*n) | Good for single path query |
| DFS | O(m*n) | O(m*n) | Recursive approach, similar to BFS |
| Dynamic Programming | O(m*n) | O(m*n) | Overkill for simple reachability |

## Reference Implementation

In [ ]:
from typing import List


class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n
    
    def find(self, x: int) -> int:
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # Path compression
        return self.parent[x]
    
    def union(self, x: int, y: int):
        root_x, root_y = self.find(x), self.find(y)
        if root_x != root_y:
            # Union by rank
            if self.rank[root_x] > self.rank[root_y]:
                self.parent[root_y] = root_x
            elif self.rank[root_x] < self.rank[root_y]:
                self.parent[root_x] = root_y
            else:
                self.parent[root_y] = root_x
                self.rank[root_x] += 1


def hasValidPath(grid: List[List[int]]) -> bool:
    m, n = len(grid), len(grid[0])
    
    # Define which directions each street type connects
    # 0: LEFT, 1: RIGHT, 2: UP, 3: DOWN
    streets = {
        1: [0, 1],      # LEFT, RIGHT
        2: [2, 3],      # UP, DOWN
        3: [0, 3],      # LEFT, DOWN
        4: [1, 3],      # RIGHT, DOWN
        5: [0, 2],      # LEFT, UP
        6: [1, 2]       # RIGHT, UP
    }
    
    # Direction vectors: LEFT, RIGHT, UP, DOWN
    directions = [(0, -1), (0, 1), (-1, 0), (1, 0)]
    # Opposite directions: RIGHT, LEFT, DOWN, UP
    opposite = [1, 0, 3, 2]
    
    def get_index(r: int, c: int) -> int:
        return r * n + c
    
    uf = UnionFind(m * n)
    
    # Build connections
    for i in range(m):
        for j in range(n):
            current_street = grid[i][j]
            current_dirs = streets[current_street]
            
            # Check each direction this street can connect to
            for dir_idx in current_dirs:
                dr, dc = directions[dir_idx]
                ni, nj = i + dr, j + dc
                
                # Check if next cell is valid
                if 0 <= ni < m and 0 <= nj < n:
                    next_street = grid[ni][nj]
                    next_dirs = streets[next_street]
                    
                    # Check if next street accepts connection from opposite direction
                    if opposite[dir_idx] in next_dirs:
                        uf.union(get_index(i, j), get_index(ni, nj))
    
    # Check if start and end are connected
    start = get_index(0, 0)
    end = get_index(m - 1, n - 1)
    return uf.find(start) == uf.find(end)

## Validation

In [ ]:
assert hasValidPath([[2,4,3],[6,5,2]]) == True
assert hasValidPath([[1,2,1],[1,2,1]]) == False
assert hasValidPath([[1,1,2]]) == False
assert hasValidPath([[1,1,1,1,1,1,3]]) == True
assert hasValidPath([[2],[2],[2],[2],[2],[2],[6]]) == True
assert hasValidPath([[4,1],[6,1]]) == True
print('All tests passed for LC 1391.')

## Complexity Analysis
- Time Complexity: O(m*n α(m*n)), where m*n is grid size and α is inverse Ackermann function.
  - Processing each cell: O(m*n)
  - Each union/find operation: O(α(m*n))
- Space Complexity: O(m*n) for Union-Find structure.
- Bottleneck: Iterating through all cells and checking directional compatibility.

## Edge Cases & Pitfalls
- Single cell grid: Start and end are the same, always valid.
- Incompatible street directions: Adjacent streets that don't align.
- Circular paths: May connect many cells but not reach the destination.
- Direction validation: Must check both sides of connection (exit and entry match).
- Grid boundaries: Careful not to access out-of-bounds cells.

## Follow-up Variants
- Find the shortest valid path length.
- Count all valid paths from start to end.
- Allow street rotation and find minimum rotations needed.
- Extend to 3D grid with additional directional constraints.

## Takeaways
- Directional constraints require careful validation of both entry and exit points.
- Union-Find efficiently handles connectivity even with complex constraints.
- Mapping street types to direction sets simplifies logic.
- Opposite direction concept is key: if A goes RIGHT to B, B must accept from LEFT.
- Connectivity problems can often use Union-Find even when paths have constraints.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| 200 | Number of Islands | Basic grid connectivity |
| 1559 | Detect Cycles in 2D Grid | Union-Find cycle detection |
| 959 | Regions Cut By Slashes | Grid Union-Find with virtual nodes |
| 130 | Surrounded Regions | Grid boundary connectivity |